# Elmatrix Studio v2 — scene-based portrait video

Run cells in order in Google Colab. This upgrade creates multiple designed cards with a subtle motion effect, scene-by-scene synthetic narration, a progress indicator and an end card. It uses no stock footage, no paid API and no YouTube credentials. Upload only the script-only job ZIP created by `prepare-cloud`. Preview the result before use. Free Colab availability is not guaranteed.


In [ ]:
import subprocess, sys
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','espeak-ng','fonts-dejavu-core'], check=True)
subprocess.run([sys.executable,'-m','pip','install','Pillow==12.3.0'], check=True)
print('Studio v2 dependencies ready.')


In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, zipfile
from google.colab import files
from IPython.display import Video, display

# Download the reviewed renderer from this repository branch.
renderer_url = 'https://raw.githubusercontent.com/braze222/ish-toolkit/feature/video-pilot/video-pilot/renderer_v2.py'
renderer_dir = Path(tempfile.mkdtemp(prefix='elmatrix-studio-v2-'))
renderer = renderer_dir / 'renderer_v2.py'
subprocess.run(['curl','-fsSL',renderer_url,'-o',str(renderer)], check=True)
if renderer.stat().st_size < 5000:
    raise RuntimeError('Renderer download looks incomplete; stop and retry.')
print('Upload exactly one cleaning-001-job.zip (or another prepare-cloud job ZIP).')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Choose exactly one job ZIP.')
name, content = next(iter(uploaded.items()))
if not name.lower().endswith('.zip') or len(content) > 1024 * 1024:
    raise ValueError('Expected one ZIP no larger than 1 MiB.')
job = renderer_dir / 'job.zip'
job.write_bytes(content)
result = renderer_dir / 'render-result-v2.zip'
subprocess.run([sys.executable,'-I',str(renderer),str(job),str(result)], check=True, timeout=2100)
with zipfile.ZipFile(result) as z:
    if sorted(z.namelist()) != ['render.json','video.mp4'] or z.testzip() is not None:
        raise RuntimeError('Result ZIP failed integrity checks.')
    preview = renderer_dir / 'preview.mp4'
    preview.write_bytes(z.read('video.mp4'))
    print('Render metadata:', z.read('render.json').decode('utf-8'))
display(Video(str(preview), embed=True))
files.download(str(result))
print('Save render-result-v2.zip in Files/xxix. Import into a NEW project directory; do not overwrite the existing render.')
print('No YouTube upload or channel authorization occurred.')
